# Report-Language Uncertainty Predicts Radiologist Spatial Disagreement
## Rule-based bag-of-words pipeline (Google Colab)

This notebook runs the full PadChest-GR pipeline using the **rule-based bag-of-words uncertainty classifier** instead of MedGemma. No GPU is required and it runs in well under a minute.

Pipeline:
1. Clone the repo from GitHub (branch `rule-bag-of-words`).
2. Install the small CPU-only dependencies.
3. Apply the hedge-word dictionary to every PadChest-GR finding sentence.
4. Compute reader-reader mask IoU.
5. Run group statistics, Mann-Whitney, bootstrap, permutation tests, per-finding control, OLS regression.
6. Show plots, tables, and 15 + 15 example sentences inline.

**Runtime:** `Runtime -> Change runtime type -> CPU` is fine (no GPU needed). Total runtime: ~30 s.

## 1. Clone the repo (rule-bag-of-words branch)

In [ ]:
REPO_URL = 'https://github.com/nprakash1/uncertaintyestimateyang.git'
REPO_DIR = '/content/uncertaintyestimateyang'
BRANCH   = 'rule-bag-of-words'

import os, subprocess, shutil
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR], check=True)

PROJECT_DIR = os.path.join(REPO_DIR, 'project')
SRC_DIR     = os.path.join(PROJECT_DIR, 'src')
print('Project dir:', PROJECT_DIR)
print('Contents:', sorted(os.listdir(PROJECT_DIR)))
print('On branch:', subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

## 2. Install dependencies (CPU only)

No transformers / torch / GPU needed.

In [ ]:
!pip install -q -U 'scipy>=1.10' 'scikit-learn>=1.3' 'matplotlib>=3.7' 'pandas>=2.0'
import pandas, scipy, sklearn, matplotlib
print('pandas', pandas.__version__, '| scipy', scipy.__version__,
      '| sklearn', sklearn.__version__, '| matplotlib', matplotlib.__version__)

## 3. Inspect the hedge-word dictionary that will be used

These are the 47 phrases the rule-based classifier looks for (case-insensitive substring match). If a sentence contains any of them, it is labeled `uncertain`; otherwise `certain`.

In [ ]:
import sys
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from medgemma_uncertainty import UNCERTAIN_TERMS
print(f'{len(UNCERTAIN_TERMS)} hedge phrases:\n')
for i, t in enumerate(UNCERTAIN_TERMS, 1):
    print(f'{i:>2}. {t}')

## 4. Run the end-to-end pipeline (rule-based)

This streams stdout/stderr live so you see progress and any traceback.

In [ ]:
import os, sys, subprocess, time

os.chdir(SRC_DIR)
print('CWD:', os.getcwd())

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

cmd = [sys.executable, '-u', 'run_pipeline.py', '--scorer', 'rule']
print('Running:', ' '.join(cmd))
t0 = time.time()

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        env=env, bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='')
rc = proc.wait()
print(f'\nTotal runtime: {time.time()-t0:.1f} s  (exit code {rc})')
assert rc == 0, f'run_pipeline.py exited with code {rc}'

## 5. Inspect the results

In [ ]:
import json, pandas as pd, pathlib

OUT = pathlib.Path(PROJECT_DIR) / 'outputs'
FIG = pathlib.Path(PROJECT_DIR) / 'figures'
PROC = pathlib.Path(PROJECT_DIR) / 'data' / 'processed'

group_stats = json.loads((OUT / 'group_statistics.json').read_text())
stat_tests  = json.loads((OUT / 'statistical_tests.json').read_text())
regression  = json.loads((OUT / 'regression_results.json').read_text())

print('=== Group statistics ===')
print(json.dumps(group_stats, indent=2))
print('\n=== Statistical tests ===')
print(json.dumps(stat_tests, indent=2))
print('\n=== Regression (control for finding label + log box area) ===')
print(json.dumps(regression, indent=2))

In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown('### Reader IoU by uncertainty group'))
display(Image(filename=str(FIG / 'iou_by_uncertainty_group.png')))

display(Markdown('### IoU histogram by group'))
display(Image(filename=str(FIG / 'iou_histogram_by_group.png')))

display(Markdown('### Example grid'))
display(Image(filename=str(FIG / 'example_grid.png')))

In [ ]:
display(Markdown('### Per-finding control analysis (>=10 in each group)'))
pf = pd.read_csv(OUT / 'per_finding_analysis.csv')
display(pf)

display(Markdown('### Top examples: uncertain + low IoU'))
display(pd.read_csv(OUT / 'examples_uncertain_low_iou.csv').head(15))

display(Markdown('### Top examples: certain + high IoU'))
display(pd.read_csv(OUT / 'examples_certain_high_iou.csv').head(15))

## 6. 15 random uncertain + 15 random certain example sentences

(Rule-based labels, with the hedge word(s) that triggered each `uncertain` classification.)

In [ ]:
import pandas as pd, pathlib, textwrap

merged = pd.read_csv(PROC / 'samples_with_uncertainty_and_iou.csv')

K = 15
RNG_SEED = 0

def show_examples(df, label, k=K, seed=RNG_SEED):
    sub = df[df['uncertainty_label'] == label]
    if len(sub) == 0:
        print(f'No {label} samples.')
        return
    sample = sub.sample(min(k, len(sub)), random_state=seed)
    print(f'\n=== {k} {label.upper()} examples (n={len(sub)} total) ===')
    for i, (_, row) in enumerate(sample.iterrows(), 1):
        sentence = textwrap.shorten(str(row['sentence']), width=140, placeholder='...')
        triggers = row.get('uncertainty_triggers', '')
        iou = float(row.get('reader_iou', float('nan')))
        finding = row.get('finding_label', '')
        print(f'{i:>2}. [{finding}]  IoU={iou:.2f}')
        print(f'    "{sentence}"')
        if isinstance(triggers, str) and triggers and triggers not in ('[]', 'nan'):
            print(f'    triggers: {triggers}')

show_examples(merged, 'uncertain')
show_examples(merged, 'certain')

## 7. Summary of the merged sample table

In [ ]:
print('rows:', len(merged))
display(merged['uncertainty_label'].value_counts())
display(merged.groupby('uncertainty_label')['reader_iou'].describe())

## 8. (Optional) Download all outputs as a zip

In [ ]:
import shutil
shutil.make_archive('/content/rule_uncertainty_outputs', 'zip',
                    root_dir=PROJECT_DIR,
                    base_dir='.')
print('Saved /content/rule_uncertainty_outputs.zip')
try:
    from google.colab import files
    files.download('/content/rule_uncertainty_outputs.zip')
except Exception:
    pass